<a href="https://colab.research.google.com/github/ADRIANVM117/data-science-portfolio/blob/main/scenario-portfolio-lab/notebooks/06_portfolio_construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

El botón abre esta notebook en Google Colab. Allí ejecuta primero la siguiente celda de preparación para obtener el código y los reportes públicos; no descarga datos de mercado ni solicita credenciales.

# 07. Evaluación temporal de las reglas de cartera

## ¿Qué habría ocurrido al construir y mantener carteras con estas reglas, utilizando únicamente la información admisible en cada decisión?

En 06 construimos una cartera candidata de mínimo CVaR en un único corte histórico. Ese resultado describe su comportamiento bajo los escenarios utilizados para optimizarla, no su desempeño posterior.

En 07 reconstruiremos decisiones sucesivas y mediremos lo que ocurre después de cada una. Compararemos tres reglas: pesos iguales, mínimo CVaR con probabilidades exponenciales y mínimo CVaR condicionado por VIX.

Conservaremos los ocho ETFs y las restricciones de 06: cartera plenamente invertida, sin posiciones cortas, margen ni efectivo, y un tope de 15% por instrumento. Los escenarios de riesgo serán inicialmente de una sesión; el calendario de rebalanceo y la convención de ejecución se definirán por separado.

El ejercicio será retrospectivo. El universo se seleccionó después de observar la historia y la muestra ya fue explorada, por lo que no representa un *holdout* intacto ni resultados de operación real.

**Siguiente pregunta:** ¿Cuándo calculamos los pesos, cuándo podemos ejecutarlos y desde qué momento les atribuimos resultados?

## 2. ¿Cuándo decidimos y desde cuándo cuentan los resultados?

Todos los desplazamientos de esta convención corresponden a sesiones del calendario publicado, no a días naturales. Separamos la información que sirve para decidir de los retornos que atribuimos después a la decisión.

| Momento | Información o acción |
| --- | --- |
| Hasta el cierre de $t-1$ | Últimos retornos utilizados para ajustar el modelo. |
| Antes de ejecutar en $t$ | Calculamos los pesos con esa historia y el VIX de $t-1$. |
| Al cierre de $t$ | Ejecutamos hipotéticamente el rebalanceo. |
| Del cierre de $t$ al cierre de $t+1$ | Primer retorno atribuible a los pesos nuevos. |

Ejecutar al cierre es un supuesto del backtest, no una garantía de obtener ese precio en una operación real.

El retorno fechado $t$ pertenece a los pesos anteriores. En la primera inversión, la evaluación comienza después de ejecutar la cartera; todavía no calculamos resultados del periodo anterior a esa inversión.

No usamos el retorno ni el VIX de $t$ para decidir los pesos ejecutados al cierre de $t$.

Consideramos disponible el VIX de $t-1$ antes de ejecutar en $t$, siguiendo la evidencia aportada por el proveedor: normalmente estaría disponible hacia las 22:00 US/Eastern. Esta es una convención retrospectiva basada en esa comunicación, no una disponibilidad histórica verificada para cada fecha ni una garantía de publicación.

El VIX tiene dos usos distintos. En la muestra histórica de ajuste, cada retorno fechado $x$ conserva su asociación con el VIX de $x-2$, según la convención de 04/05. El VIX de $t-1$ identifica el estado conocido para la decisión actual. No cambiaremos la alineación histórica para hacer coincidir ambas fechas.

Las tres reglas —pesos iguales, mínimo CVaR exponencial y mínimo CVaR condicionado por VIX— compartirán la misma convención de decisión, ejecución y atribución de resultados.

Como ejemplo conceptual, considere tres sesiones consecutivas. En la sesión anterior termina la información utilizada; en la sesión de ejecución aplicamos los nuevos pesos al cierre; en la sesión siguiente observamos el primer retorno de los nuevos pesos. Este ejemplo no presupone precios, pesos ni rendimientos concretos.

**Siguiente pregunta:** ¿En qué sesiones rebalancearemos y cuánta historia exigiremos antes de invertir?